# Executive Payment Operations Dashboard
# RebTech Academy Submission_05
## Payment Transaction Analytics

This notebook presents an executive-level analysis of payment transaction data using
Python, Pandas, and Plotly.

The objective is to transform cleaned transaction-level data into business-oriented
Key Performance Indicators (KPIs), trends, channel analysis, payment-status analysis,
and transaction-level insights.

The analysis follows the workflow:

**Clean Dataset → Filtering → KPI Calculation → Trend Analysis → Channel Analysis → Transaction Drill-Down → Business Insights**

## 1. Introduction

A payment operations dashboard converts transaction-level data into a compact set of
business metrics that can support monitoring, decision-making, and operational analysis.

While exploratory data analysis focuses on understanding the characteristics of the
dataset, an executive dashboard focuses on answering practical business questions.

### Key questions addressed

This dashboard investigates:

1. How much gross revenue was generated?
2. How much revenue remained after settlement adjustments?
3. What is the average order value?
4. What percentage of transactions were successful?
5. How much money was refunded?
6. How does revenue change over time?
7. Which payment channels contribute the most revenue?
8. How does payment status vary across payment methods?
9. How is net settlement distributed across payment channels?
10. Which individual transactions require further investigation?

### Analytical approach

The dashboard uses filtering dimensions such as:

- Year
- Payment method
- Payment status

These filters allow the same business metrics to be examined for different subsets
of the transaction population.

## 2. Business Objective

The primary objective is to provide an executive-level view of payment operations.

The dashboard combines financial, operational, and transaction-level indicators.

### KPI Categories

| Category | KPI / Analysis |
|---|---|
| Revenue | Gross Revenue |
| Settlement | Net Revenue |
| Customer Transaction Value | Average Order Value (AOV) |
| Payment Operations | Success Rate |
| Refund Operations | Total Refunds |
| Time Analysis | Monthly Revenue & Settlement Trend |
| Channel Analysis | Revenue Share by Payment Method |
| Payment Operations | Status Distribution by Method |
| Distribution Analysis | Net Settlement Distribution |
| Transaction Analysis | Transaction Drill-Down |

The purpose is not only to calculate these metrics, but also to understand what
they reveal about the operational performance of the payment system.

## 3. Import Libraries

The analysis requires libraries for data manipulation, numerical analysis,
visualization, and interactive plotting.

### Libraries used

- **Pandas** — data loading, filtering, grouping, and aggregation.
- **NumPy** — numerical operations.
- **Matplotlib** — general-purpose plotting support.
- **Seaborn** — statistical visualization support.
- **Plotly Express** — interactive business visualizations.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from IPython.display import display

## 4. Visualization Configuration

Consistent visualization settings improve readability and make different charts
easier to compare.

Plotly is used for the main dashboard-style visualizations because it provides
interactive charts such as hover information, zooming, and legend controls.

In [2]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11})

## 5. Load the Clean Dataset

The analysis uses the cleaned payment transaction dataset generated during the
previous data-cleaning stage.

The dataset is loaded from a relative path so that the same file structure can
also be used when the project is deployed as a Streamlit application.

The `payment_date` field is explicitly converted to Pandas datetime format so
that it can be used reliably for time-based analysis.

In [3]:
df = pd.read_csv("clean_dataset.csv")

df['payment_date'] = pd.to_datetime(df['payment_date'])

print("Clean dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Clean dataset loaded successfully.
Rows: 90,000
Columns: 15


## 5.1 Initial Dataset Inspection

Before calculating business metrics, it is important to verify the structure
and contents of the dataset.

The following checks identify:

- Number of observations
- Number of variables
- Available payment methods
- Available payment statuses
- Date range
- Basic data structure

In [4]:
print("Dataset shape:", df.shape)

print("\nPayment Methods:")
display(pd.DataFrame({
    "Payment Method": sorted(df['payment_method'].dropna().unique())
}))

print("\nPayment Statuses:")
display(pd.DataFrame({
    "Payment Status": sorted(df['payment_status'].dropna().unique())
}))

print("\nDate Range:")
print("Start:", df['payment_date'].min())
print("End:", df['payment_date'].max())

Dataset shape: (90000, 15)

Payment Methods:


,Payment Method
0,Cardless Emi
1,Cash On Delivery
2,Credit Card
3,Debit Card
4,Emi
5,Gift Card
6,Net Banking
7,Pay Later
8,Unknown
9,Upi



Payment Statuses:


,Payment Status
0,Failed
1,Not Charged
2,Pending
3,Refunded
4,Success



Date Range:
Start: 2023-01-02 00:00:00
End: 2025-12-31 00:00:00


## 6. Interactive Filtering Logic

An executive dashboard should allow users to analyze different subsets of the
transaction population.

The original Streamlit dashboard provides three filters:

1. **Year**
2. **Payment Method**
3. **Payment Status**

The filtering logic is:

**Filtered Dataset = Year Filter AND Payment Method Filter AND Payment Status Filter**

Only records satisfying all selected conditions are included in the subsequent
KPI and visualization calculations.

In Streamlit, these selections are made interactively through sidebar widgets.

In this Jupyter Notebook, the same logic is reproduced using explicitly defined
filter variables.

In [5]:
# Select the years to include
year_filter = sorted(df['order_year'].dropna().unique())

# Select the payment methods to include
method_filter = sorted(df['payment_method'].dropna().unique())

# Select the payment statuses to include
status_filter = sorted(df['payment_status'].dropna().unique())

filtered_df = df[
    (df['order_year'].isin(year_filter)) &
    (df['payment_method'].isin(method_filter)) &
    (df['payment_status'].isin(status_filter))
]

print(f"Original records: {len(df):,}")
print(f"Filtered records: {len(filtered_df):,}")

Original records: 90,000
Filtered records: 90,000


## 7. Executive KPI Analysis

Key Performance Indicators (KPIs) are selected measurements used to summarize
important aspects of business performance.

For payment operations, a single revenue number is not sufficient.

For example:

- High gross revenue does not necessarily mean high net revenue.
- High average transaction value does not necessarily mean high payment success.
- High revenue may be accompanied by significant refunds.
- A payment channel may generate substantial revenue but have a different
  operational success profile.

Therefore, multiple KPIs should be interpreted together.

The dashboard calculates five executive KPIs:

1. Gross Revenue
2. Net Revenue
3. Average Order Value (AOV)
4. Payment Success Rate
5. Total Refunds Issued

## 7.1 KPI Calculation

The following definitions are used.

### Gross Revenue

Gross revenue represents the total amount paid across the filtered transactions.

\[
Gross\ Revenue = \sum Amount\ Paid
\]

### Net Revenue

Net revenue represents the total net settlement amount recorded in the dataset.

\[
Net\ Revenue = \sum Net\ Settlement\ Amount
\]

### Average Order Value

Average Order Value (AOV) represents the average amount paid per transaction.

\[
AOV = \frac{\sum Amount\ Paid}{Number\ of\ Transactions}
\]

### Payment Success Rate

The payment success rate measures the proportion of filtered transactions whose
payment status is recorded as `Success`.

\[
Success\ Rate =
\frac{Successful\ Transactions}
{Total\ Transactions}
\times 100
\]

### Total Refunds

Total refunds represent the sum of recorded refund amounts.

\[
Total\ Refunds = \sum Refund\ Amount
\]

In [6]:
gross_revenue = filtered_df['amount_paid'].sum()

net_revenue = filtered_df['net_settlement_amount'].sum()

aov = (
    filtered_df['amount_paid'].mean()
    if len(filtered_df) > 0
    else 0
)

success_rate = (
    len(filtered_df[filtered_df['payment_status'] == 'Success'])
    / len(filtered_df) * 100
    if len(filtered_df) > 0
    else 0
)

refunds = filtered_df['refund_amount'].sum()

## 7.2 Executive KPI Scorecard

The calculated values are presented below as an executive summary.

These KPIs provide a high-level view of the financial and operational condition
of the selected transaction population.

In [7]:
kpi_summary = pd.DataFrame({
    'KPI': [
        'Gross Revenue',
        'Net Revenue',
        'Average Order Value (AOV)',
        'Payment Success Rate',
        'Total Refunds Issued'
    ],
    'Value': [
        f"₹{gross_revenue:,.0f}",
        f"₹{net_revenue:,.0f}",
        f"₹{aov:,.2f}",
        f"{success_rate:.1f}%",
        f"₹{refunds:,.0f}"
    ]
})

display(kpi_summary)

,KPI,Value
0,Gross Revenue,"₹1,986,733,131"
1,Net Revenue,"₹1,785,996,888"
2,Average Order Value (AOV),"₹22,074.81"
3,Payment Success Rate,72.3%
4,Total Refunds Issued,"₹194,097,940"


## 8. Monthly Revenue & Net Settlement Trend

Time-series analysis examines how a metric changes over time.

Monthly aggregation is useful because transaction-level data can contain
thousands of individual records, making it difficult to identify overall
financial trends directly.

The dashboard compares:

- Monthly Amount Paid
- Monthly Net Settlement Amount

This allows us to investigate:

- Revenue growth or decline
- Seasonal patterns
- Periods of unusually high or low activity
- Differences between gross payment volume and net settlement

In [8]:
monthly_trend = (
    filtered_df
    .groupby(['order_year', 'order_month'])
    [['amount_paid', 'net_settlement_amount']]
    .sum()
    .reset_index()
)

monthly_trend['Period'] = (
    monthly_trend['order_year'].astype(str)
    + "-"
    + monthly_trend['order_month'].astype(str).str.zfill(2)
)

display(monthly_trend)

,order_year,order_month,amount_paid,net_settlement_amount,Period
0,2023,1,7.834162e+06,7.441645e+06,2023-01
1,2023,2,1.930686e+07,1.697530e+07,2023-02
2,2023,3,3.499803e+07,3.138925e+07,2023-03
3,2023,4,4.223029e+07,3.814877e+07,2023-04
4,2023,5,5.176673e+07,4.497673e+07,2023-05
5,2023,6,5.340808e+07,4.725580e+07,2023-06
6,2023,7,5.617908e+07,5.096211e+07,2023-07
7,2023,8,5.459432e+07,4.843693e+07,2023-08
8,2023,9,5.101992e+07,4.637382e+07,2023-09
9,2023,10,5.270690e+07,4.688402e+07,2023-10


In [9]:
fig_trend = px.area(
    monthly_trend.sort_values('Period'),
    x='Period',
    y=['amount_paid', 'net_settlement_amount'],
    title='Monthly Revenue & Net Settlement Trend',
    labels={
        'value': 'Amount (₹)',
        'Period': 'Period',
        'variable': 'Metric'
    },
    color_discrete_sequence=['#1f77b4', '#2ca02c']
)

fig_trend.show()

### Interpretation

The trend should be examined for:

- Increasing or decreasing revenue over time
- Peaks in payment volume
- Periods where net settlement diverges substantially from gross revenue
- Repeated seasonal patterns
- Unusual monthly changes requiring further investigation

The visual trend should be interpreted together with the underlying transaction
data rather than treated as proof of a specific business cause.

## 9. Revenue Share by Payment Channel

Payment methods represent different transaction channels through which customers
complete payments.

Revenue share analysis determines how much of the total recorded payment value
is associated with each payment method.

The calculation is based on:

\[
Revenue\ Share =
\frac{Revenue\ from\ Payment\ Method}
{Total\ Revenue}
\times 100
\]

This helps identify the channels that contribute the largest portion of payment
volume.

In [10]:
channel_revenue = (
    filtered_df
    .groupby('payment_method')['amount_paid']
    .sum()
    .reset_index()
)

channel_revenue['Revenue Share (%)'] = (
    channel_revenue['amount_paid']
    / channel_revenue['amount_paid'].sum()
    * 100
)

display(channel_revenue.sort_values(
    'amount_paid',
    ascending=False
))

,payment_method,amount_paid,Revenue Share (%)
8,Unknown,8.079935e+08,40.669454
9,Upi,3.834410e+08,19.300075
1,Cash On Delivery,1.989114e+08,10.011984
2,Credit Card,1.801016e+08,9.065215
3,Debit Card,1.507917e+08,7.589933
6,Net Banking,1.027048e+08,5.169530
10,Wallet,7.741945e+07,3.896822
4,Emi,4.264927e+07,2.146704
7,Pay Later,1.960756e+07,0.986925
5,Gift Card,1.189851e+07,0.598898


In [11]:
fig_pie = px.pie(
    filtered_df,
    names='payment_method',
    values='amount_paid',
    hole=0.4,
    title='Revenue Share by Payment Channel',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig_pie.show()

## 10. Payment Status Distribution by Method

Payment status analysis examines how transaction outcomes differ across payment
methods.

Instead of comparing only raw transaction counts, the analysis normalizes each
payment method to 100%.

Therefore, each payment method represents its own percentage distribution of
statuses.

This makes channels with different transaction volumes directly comparable.

The calculation uses:

\[
Status\ Percentage =
\frac{Transactions\ with\ a\ given\ status}
{Total\ Transactions\ for\ that\ payment\ method}
\times 100
\]

In [12]:
status_ct = (
    pd.crosstab(
        filtered_df['payment_method'],
        filtered_df['payment_status'],
        normalize='index'
    ) * 100
)

display(status_ct)

payment_status,Failed,Not Charged,Pending,Refunded,Success
payment_method,,,,,
Cardless Emi,6.457926,0.000000,13.698630,8.414873,71.428571
Cash On Delivery,0.000000,9.916470,12.722210,6.039837,71.321482
Credit Card,5.081201,0.000000,12.549213,10.433071,71.936516
Debit Card,5.545072,0.000000,11.946582,10.277254,72.231093
Emi,5.400284,0.000000,11.700616,9.805779,73.093321
Gift Card,5.996473,0.000000,11.463845,10.052910,72.486772
Net Banking,5.297273,0.000000,11.756817,9.767546,73.178364
Pay Later,5.912334,0.000000,12.742100,9.072375,72.273191
Unknown,4.407683,1.579327,12.132230,9.462085,72.418674


In [13]:
fig_bar = px.bar(
    status_ct.reset_index(),
    x='payment_method',
    y=status_ct.columns,
    barmode='stack',
    title='Payment Status Distribution by Method (%)',
    labels={
        'value': 'Percentage (%)',
        'payment_method': 'Payment Method'
    }
)

fig_bar.show()

## 11. Net Settlement Distribution by Payment Method

A box plot summarizes the distribution of a numerical variable across different
categories.

In this analysis, the numerical variable is:

**Net Settlement Amount**

and the categorical variable is:

**Payment Method**

The box plot allows comparison of:

- Median
- Interquartile range
- Distribution spread
- Potential extreme observations

The original dashboard applies:

```python
net_settlement_amount > 0

## 12. Transaction Record Drill-Down

Executive KPIs and aggregated charts provide a high-level view of operations,
but business analysis may also require examination of individual transactions.

The drill-down table exposes selected transaction-level fields including:

- Payment ID
- Order ID
- Payment Date
- Payment Method
- Payment Status
- Amount Paid
- Net Settlement Amount
- Profitability Status

Displaying individual records allows analysts to investigate unusual or
business-critical transactions identified during the aggregate analysis.

In [14]:
drill_down_columns = [
    'payment_id',
    'order_id',
    'payment_date',
    'payment_method',
    'payment_status',
    'amount_paid',
    'net_settlement_amount',
    'profitability_status'
]

display(
    filtered_df[drill_down_columns].head(100)
)

,payment_id,order_id,payment_date,payment_method,payment_status,amount_paid,net_settlement_amount,profitability_status
0,PAY1000000,ORD1000000,2023-02-12,Emi,Failed,0.00,0.00,Break-Even
1,PAY1000001,ORD1000001,2023-02-12,Upi,Success,4998.00,4998.00,Profitable
2,PAY1000002,ORD1000002,2024-12-14,Net Banking,Success,867.30,867.30,Profitable
3,PAY1000003,ORD1000003,2024-12-14,Cash On Delivery,Not Charged,0.00,0.00,Break-Even
4,PAY1000005,ORD1000005,2025-01-08,Cash On Delivery,Success,8275.89,8275.89,Profitable
...,...,...,...,...,...,...,...,...
95,PAY1000107,ORD1000107,2025-08-06,Unknown,Success,8275.89,8275.89,Profitable
96,PAY1000109,ORD1000109,2025-08-06,Unknown,Refunded,9675.38,0.00,Break-Even
97,PAY1000110,ORD1000110,2024-02-26,Credit Card,Success,3204.60,3156.53,Profitable
98,PAY1000112,ORD1000112,2025-11-24,Unknown,Refunded,13125.00,0.00,Break-Even


## 13. Export Filtered Transaction Data

The filtered transaction dataset can be exported as a CSV file for further
analysis or operational review.

Only the records satisfying the selected filters are exported.

In [15]:
filtered_df.to_csv(
    "filtered_transactions.csv",
    index=False
)

print("Filtered transaction data exported successfully.")

Filtered transaction data exported successfully.


## 14. Key Business Insights

The dashboard should be interpreted as a combination of financial,
operational, channel, and transaction-level evidence.

### 1. Revenue Performance

Gross revenue indicates the total payment value recorded in the selected
transaction population, while net revenue reflects the corresponding
net settlement amount.

A substantial difference between these values may indicate the influence
of fees, refunds, adjustments, or other settlement effects represented in
the dataset.

### 2. Customer Transaction Value

Average Order Value (AOV) indicates the average payment amount per transaction.

AOV should be interpreted together with transaction volume because a high
average transaction value does not necessarily imply high overall business
activity.

### 3. Payment Operations

Payment Success Rate provides an operational indicator of how frequently
transactions reach the `Success` status.

Comparing this measure across payment methods can help identify channels
that may warrant additional operational investigation.

### 4. Refund Exposure

Total Refunds Issued represents the monetary value of recorded refunds.

A high refund amount relative to gross revenue may warrant investigation into
refund frequency, transaction categories, payment channels, or underlying
business processes.

### 5. Channel Concentration

Revenue Share by Payment Channel identifies which payment methods contribute
the largest proportion of recorded payment value.

High concentration in a small number of channels may indicate dependence on
those channels and may therefore be relevant to payment operations planning.

# 15. Conclusion

This notebook transformed cleaned payment transaction data into an
executive-level analytical view.

The analysis combined:

- Data preparation
- Business KPI calculation
- Monthly trend analysis
- Payment-channel analysis
- Payment-status analysis
- Distribution analysis
- Transaction-level drill-down

The dashboard demonstrates how transaction-level records can be converted
into decision-oriented information.

Importantly, the visualizations should not be interpreted independently.
Revenue, settlement, success rate, refunds, payment-method distribution,
and transaction-level observations should be considered together when
evaluating payment operations.

The analysis therefore provides a foundation for further business questions,
such as identifying underperforming payment channels, investigating unusual
settlement values, and monitoring changes in payment performance over time.

st.set_page_config()
st.sidebar.multiselect()
st.columns()
st.metric()
st.plotly_chart()
st.dataframe()
st.download_button()

### Streamlit Dashboard Functions
- `st.set_page_config()` → sets app title, icon, layout
- `st.sidebar.multiselect()` → creates filter widgets
- `st.columns()` → arranges KPI cards side by side
- `st.metric()` → displays KPI values
- `st.plotly_chart()` → embeds interactive charts
- `st.dataframe()` → shows a scrollable table
- `st.download_button()` → lets users export filtered data
